Here, we will implement our first neural network! You will need to accomplish a number of tasks:

1. Implement a linear layer in `linear` (15 points)
2. Implement a perceptron (single layer neural network) in `f` (15 points)
3. Implement the `loss_single` function, which is the loss function for a single $x_{[i]}, y_{[i]}$ datapoint. (15 points)

Then, you will need to use what we learned about optimization and backpropagation. You must

4. Implement the `single_grad_loss`, which computes the gradient of the `loss_single` function. Written mathematically, you must compute $\nabla_{\mathbf{\theta}} \mathcal{L}(\mathbf{x}, y, (\theta_0, \theta_1, \theta_2))$. You may NOT use `jax.grad`. You need to figure out the gradient by hand. (25 points)
5. Ensure your functions are correct, such that the error computed between your `single_grad_loss` and the `jax.grad` function is less than 0.001. (30 points)

This assignment will be graded out of 100 points.

In [ ]:
import jax
import jax.numpy as jnp
import operator
import matplotlib.pyplot as plt

# Set up some variables for our experiments
n = 32 # number of x_[i] y_[i] to train
d_x = 2 # size of x
d_y = 1 # size of y

In [ ]:
# Define our neural network

def sigmoid(z):
  '''Activation function, a differentiable approximation of the step function'''
  return 1 / (1 + jnp.exp(-z))

def linear(x, theta):
  '''Linear layer'''
  # TODO: Implement a linear layer
  return jnp.zeros_like(theta[0][1])

def f(x, theta):
  '''A single-layer perceptron (neural network)'''
  # TODO: Create a single-layer neural network (perceptron)
  # Also known as a wide neural network
  return jnp.zeros_like(theta[1])

def debug_shape(x):
  '''Helpful function for printing tensor shapes'''
  print(jax.tree.map(lambda x: x.shape, x))

In [ ]:
# Define our losses

def loss_single(x, y, theta):
  '''The loss function for a single datapoint x_i, y_i'''
  # TODO: Define the square error loss function for a single x_i, y_i
  return jnp.array(1.0)

def loss(x, y, theta):
  '''The loss function as defined over a dataset'''
  # vmap is an extremely powerful tool in jax, that allows us to parallelize
  # across any dimension. In this case, we can make f, which operates over
  # a single input, work with multiple inputs. Read the documentation for vmap!
  # Vectorize across the first two arguments, but not the parameters
  multiple_loss = jax.vmap(loss_single, in_axes=(0, 0, None))
  return jnp.sum(multiple_loss(x, y, theta))


Next, you will need to implement the gradient of the loss function with respect to the parameters, $\frac{\partial \mathcal{L}}{\partial \mathbf{\theta}}$. Using the chain rule, we can decompose this into a number of functions.

In [ ]:
# Next, implement the gradient of the loss function with respect to the parameters
# It will be very helpful to write the full loss function down on paper, then
# take the derivatives using the chain rule.
#
# First, write down f
# Then, write down the loss function with respect to f
# Then differentiate!

def single_grad_loss(x, y, theta):
  '''Compute the gradient with respect to theta for a single datapoint x_i, y_i'''
  # TODO: Find [dL/dtheta_0, dL/dtheta_1, dL/dtheta_2]
  # Hint: Remember the chain rule
  W, b = theta

  # You will need these values to compute the gradient
  z = linear(x, theta)
  y_pred = sigmoid(z)

  # Compute gradients here for the weight and bias
  # it should be a tuple of this shape
  grad = jnp.ones((d_y,)), jnp.ones((d_x,))


  return grad

def grad_loss(x, y, theta):
  # Compute the gradient over multiple x_i, y_i
  grad = jax.vmap(single_grad_loss, in_axes=(0, 0, None))(x, y, theta)
  # Sum the gradient over all the samples
  return grad[0].sum(axis=0, keepdims=True), grad[1].sum(axis=0)

In [ ]:
# Generate our datasets X and Y
def generate_regression_data(key, n_samples, noise=0.1):
    key, subkey = jax.random.split(key)
    X = jax.random.uniform(subkey, (n_samples, d_x), minval=-1, maxval=1)  # Random points in the range [-1, 1]
    y = jnp.sqrt(X[:, 0]**2 + X[:, 1]**2) + noise * jax.random.normal(key, (n_samples,))  # Distance from origin with noise
    y = (y - y.min()) / (y.max() - y.min())  # Normalize to [0, 1]
    return X, y

x, y = generate_regression_data(jax.random.PRNGKey(0), n)

# Randomly initialize the parameters of our neural network
theta = (
    # Weights
    jax.random.uniform(jax.random.PRNGKey(1), (d_x, d_y)) - 0.5,
    # Bias
    jax.random.uniform(jax.random.PRNGKey(2), (d_y,)) - 0.5
)

In [ ]:
# Let's verify your gradient is correct!
# jax can compute the gradient automatically

# First ensure that f does not crash...
f(x[0], theta)

# Let's make sure the loss functions do not crash
loss_single(x[0], y[0], theta)
loss(x, y, theta)

# Determine if our gradient is correct for a single sample
# NOTE: We determine correctness by differentiating your loss function
# If your loss function is incorrect, grad(loss) will also be incorrect!
# Differentiate f with respect to the first argument (theta)
your_grad_single = single_grad_loss(x[0], y[0], theta)
# Jax can automatically differentiate functions
# By setting argnums=2, we differentiate with respect to the second argument (theta)
auto_grad_single = jax.grad(loss_single, argnums=2)(x[0], y[0], theta)
# Get rid of singleton dimension
auto_grad_single = jax.tree.map(lambda a: a.reshape(-1), auto_grad_single)

print(f"Automatic gradient:\n {auto_grad_single}")
print(f"Your gradient:\n {your_grad_single}")

grad_difference = jax.tree.map(lambda a, b: a - b, auto_grad_single, your_grad_single)
print(f"Difference in gradient", grad_difference)

grad_error = jnp.linalg.norm(jnp.concatenate(grad_difference))
print(f"Gradient error", grad_error)
if grad_error < 1e-3:
  print("Gradient correct!")
else:
  print("Gradient incorrect!")

Automatic gradient:
 (Array([0., 0.], dtype=float32), Array([0.], dtype=float32))
Your gradient:
 (Array([1., 1.], dtype=float32), Array([1.], dtype=float32))
Difference in gradient (Array([-1., -1.], dtype=float32), Array([-1.], dtype=float32))
Gradient error 1.7320508
Gradient incorrect!


In [ ]:
# Determine if our gradient is correct over an entire dataset
# NOTE: We determine correctness by differentiating your loss function
# If your loss function is incorrect, grad(loss) will also be incorrect!
# Differentiate f with respect to the first argument (theta)
your_grad = grad_loss(x, y, theta)
auto_grad = jax.grad(loss, argnums=2)(x, y, theta)
# Get rid of singleton dimension
auto_grad = jax.tree.map(lambda a: a.reshape(-1), auto_grad)
# Reduce across samples

print(f"Automatic gradient:\n {auto_grad}")
print(f"Your gradient:\n {your_grad}")

grad_difference = jax.tree.map(lambda a, b: a - b, auto_grad, your_grad)
print(f"Difference in gradient", grad_difference)

grad_error = jnp.linalg.norm(jnp.concatenate(jax.tree.map(lambda x: x.reshape(-1), grad_difference)))
print(f"Gradient error", grad_error)
if grad_error < 1e-3:
  print("Gradient correct!")
else:
  print("Gradient incorrect!")

Automatic gradient:
 (Array([0., 0.], dtype=float32), Array([0.], dtype=float32))
Your gradient:
 (Array([[32., 32.]], dtype=float32), Array([32.], dtype=float32))
Difference in gradient (Array([[-32., -32.]], dtype=float32), Array([-32.], dtype=float32))
Gradient error 55.425625
Gradient incorrect!
